# Extensions

Live mode recomputes reduced reversal, multi-timescale, WM/STC, device-TD, and beta-sensitivity diagnostics.

### Setup and Dependencies
Imports the trace package, plotting utilities, and configures the default execution mode.


In [ ]:
import os, math, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

RESULT_MODE = "live"
if RESULT_MODE not in {"live", "full_sweep_cache"}:
    raise ValueError("RESULT_MODE must be 'live' or 'full_sweep_cache'")
GREEN, INDIGO, RED, GOLD, GREY, PURPLE, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#b07cc6", "#2b2b2b"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

def _cache(name):
    print(f"FULL-SWEEP CACHE: {paths.results_dir() / name}")
    return paths.load_result(name)

def _mean_ci(a):
    a = np.asarray(a, float); lo, hi = bootstrap_ci(a)
    return float(a.mean()), lo, hi

print("RESULT_MODE:", RESULT_MODE)
print("data/results:", paths.results_dir())

from mrl_trace.bandit import run_reversal, _summarize_reversal
from mrl_trace.extensions import load_measured_tau, run_multitimescale, wm_isolated, run_wm_stc, run_device_td, dmax_law

def _series(name, y, min_len=2):
    arr = np.asarray(y, float).ravel()
    if arr.size < min_len or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has insufficient live data for plotting: n={arr.size}")
    return arr

def _values(name, y):
    arr = np.asarray(y, float).ravel()
    if arr.size == 0 or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has no finite live values for plotting")
    return arr

def _smooth(y, win=50):
    arr = _series("curve", y, min_len=2)
    win = int(win)
    if arr.size < max(5, win):
        return arr
    left = win // 2
    right = win - 1 - left
    padded = np.pad(arr, (left, right), mode="edge")
    kernel = np.ones(win, dtype=float) / float(win)
    return np.convolve(padded, kernel, mode="valid")


### Reversal Learning
Evaluates the flexibility of the device trace to abruptly switch reward associations mid-training.


In [ ]:
if RESULT_MODE == "live":
    conds = ["device", "abstract", "no_trace"]
    raw = {c: run_reversal(c, B=4, trials=1200, n_phases=2, tau_leak=10.0, D=5.0) for c in conds}
    r = _summarize_reversal(raw, conds, trials=1200, crit=0.75, chance=0.5)
    src = "LIVE reduced reversal: 4 seeds, 1200 trials, D=5"
else:
    r = _cache("exp18_reversal.npy"); src = "full-sweep cache"
fig, ax = plt.subplots(figsize=(7.2, 3.8))
for c, col in zip(r["curves"].keys(), [GREEN, INDIGO, GREY, RED]):
    ax.plot(_smooth(r["curves"][c], win=50), color=col, lw=1.5, label=c)
ax.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); ax.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
ax.set_xlabel("trial"); ax.set_ylabel("reward rate"); ax.set_ylim(0.25, 1.05)
ax.set_title(f"Reversal learning diagnostic [{src}]"); ax.legend(frameon=False, fontsize=8); _clean(ax); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("criteria:", r.get("criteria", {}))

### Multi-Timescale Extraction
Builds the cumulative memory profile from an ensemble of heterogeneous device retention timescales.


In [ ]:
def _live_multitimescale():
    tau_pool, tau_src = load_measured_tau()
    tau_pool = tau_pool[:8]
    best_tau = float(np.median(tau_pool))
    out = {}
    for label, kind, norm in [("hetero_raw", "hetero_measured", "none"),
                              ("hetero_homeo", "hetero_measured", "homeo"),
                              ("best_single", "best_single", "none"),
                              ("no_trace", "no_trace", "none")]:
        rw, grp, *_ = run_multitimescale(kind, B=4, trials=500, tau_pool=tau_pool, tau_arg=best_tau, elig_norm=norm)
        out[label] = rw[:, -40:].mean(1)
    return {"finals": out, "tau_source": tau_src, "seeds": 1, "trials": 80}
if RESULT_MODE == "live":
    r = _live_multitimescale(); src = "LIVE oriented: measured tau subset"
else:
    r = _cache("exp19_multitimescale.npy"); src = "full-sweep cache"
finals = r.get("finals", r)
labels = list(finals.keys()); vals = [np.asarray(finals[k], float).mean() for k in labels]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.bar(np.arange(len(labels)), vals, color=[GREEN, GOLD, INDIGO, GREY][:len(labels)])
ax.axhline(0.5, ls="--", color=RED, lw=1.0); ax.axhline(0.75, ls=":", color=GREY, lw=1.0)
ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("final reward rate"); ax.set_title(f"Multi-timescale diagnostic [{src}]")
_clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("tau source:", r.get("tau_source", r.get("tau_src", "cache")))

### Consolidation and Working Memory
Models the transfer from transient working memory to short-term consolidated weights.


In [ ]:
if RESULT_MODE == "live":
    r = {
        "wm": {t: wm_isolated(t, B=4, trials=500, D_wm=1.0) for t in (1.3, 6.0)},
        "two_device": run_wm_stc(6.0, 6.0, B=4, trials=500, D_wm=1.0, D_stc=1.0)[:, -40:].mean(1),
        "shared_device": run_wm_stc(6.0, 6.0, B=4, trials=500, D_wm=1.0, D_stc=1.0, shared_device=True)[:, -40:].mean(1),
    }
    src = "LIVE reduced WM/STC: 4 seeds, 500 trials"
else:
    r = _cache("exp22_wm_stc.npy"); src = "full-sweep cache"
fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.5, 3.7))
if "wm" in r:
    taus = list(r["wm"].keys()); vals = [np.asarray(r["wm"][t], float).mean() for t in taus]
else:
    taus = list(r.get("tau", [])); vals = list(np.asarray(r.get("wm", []), float))
axA.bar(np.arange(len(taus)), vals, color=GREEN)
axA.axhline(0.5, ls="--", color=RED, lw=1.0); axA.axhline(0.75, ls=":", color=GREY, lw=1.0)
axA.set_xticks(np.arange(len(taus))); axA.set_xticklabels([f"tau={t:g}" for t in taus]); axA.set_ylim(0, 1.05)
axA.set_ylabel("accuracy"); axA.set_title("Working-memory hold")
labels = ["two device", "shared device"]
vals = [np.asarray(r.get("two_device", r.get("two", [np.nan])), float).mean(),
        np.asarray(r.get("shared_device", r.get("shared", [np.nan])), float).mean()]
axB.bar(np.arange(2), vals, color=[INDIGO, GOLD])
axB.axhline(0.5, ls="--", color=RED, lw=1.0); axB.axhline(0.75, ls=":", color=GREY, lw=1.0)
axB.set_xticks(np.arange(2)); axB.set_xticklabels(labels); axB.set_ylim(0, 1.05); axB.set_ylabel("reward rate")
axB.set_title("WM and STC substrate")
_clean(axA); _clean(axB); fig.suptitle(f"WM/STC diagnostic [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")

### Device Temporal Difference (TD)
Implements and scores the explicit TD-learning architecture using the device membrane.


In [ ]:
if RESULT_MODE == "live":
    r = {s: run_device_td(s, L=3, B=4, episodes=300, D=0.8, step_dur=0.2)[0][:, -40:].mean(1)
         for s in ("reinforce", "td_actor_critic", "td_no_homeo", "no_trace")}
    src = "LIVE oriented device-TD: 4 seeds, 300 episodes"
else:
    r = _cache("exp23_device_td.npy"); src = "full-sweep cache"
if "finals" in r:
    finals = r["finals"]
else:
    finals = r
labels = list(finals.keys()); vals = [np.asarray(finals[k], float).mean() for k in labels]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.bar(np.arange(len(labels)), vals, color=[GREEN, INDIGO, GOLD, GREY][:len(labels)])
ax.axhline(0.5, ls="--", color=RED, lw=1.0); ax.axhline(0.75, ls=":", color=GREY, lw=1.0)
ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("goal reward rate"); ax.set_title(f"Device-native TD diagnostic [{src}]")
_clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")

### Beta Parameter Sensitivity
Checks the robustness of the core learning metrics against variations in the $\beta$ decay parameter.


In [ ]:
if RESULT_MODE == "live":
    r = {"dmax": dmax_law([1.0, 0.85], seeds=4, episodes=200), "betas": [1.0, 0.85], "verdict": "live-oriented reduced Dmax sensitivity: 4 seeds, 200 episodes"}
    src = "LIVE oriented beta sensitivity: 4 seeds, 200 episodes"
else:
    r = _cache("exp21_beta_sensitivity.npy"); src = "full-sweep cache"
dmax = r.get("dmax", {})
betas = list(dmax.keys())
fig, ax = plt.subplots(figsize=(6.8, 3.8))
for b, c in zip(betas, [GREEN, INDIGO, GOLD]):
    row = dmax[b]; ax.plot(row["taus"], row["dmax"], "-o", color=c, label=f"beta={b}")
ax.set_xlabel("tau_leak (s)"); ax.set_ylabel("Dmax (s)"); ax.set_title(f"Dispersive leak sensitivity [{src}]")
ax.legend(frameon=False, fontsize=8); _clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print(r.get("verdict", ""))
# Full-scale regeneration:
# python -m mrl_trace.bandit --exp18 --full
# python -m mrl_trace.extensions --exp19 --exp21 --exp22 --exp23 --full